[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/solutions/57_depthwise_conv_solution.ipynb)

# 🟡 Solution: Depthwise Separable Convolution

Reference solution for depthwise separable convolution: a depthwise conv per channel followed by a 1×1 pointwise conv.

$$\text{out} = \text{PW}(\text{DW}(x))$$

In [ ]:
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass

In [ ]:
import torch

In [ ]:
# ✅ SOLUTION

def depthwise_separable_conv(x, dw_weight, pw_weight):
    B, C_in, H, W = x.shape
    kH, kW = dw_weight.shape[2], dw_weight.shape[3]
    patches = x.unfold(2, kH, 1).unfold(3, kW, 1)
    dw_out = (patches * dw_weight[:, 0].view(1, C_in, 1, 1, kH, kW)).sum(dim=(-2, -1))
    out = torch.einsum('bchw,oc->bohw', dw_out, pw_weight[:, :, 0, 0])
    return out

In [ ]:
torch.manual_seed(0)
B, C_in, H, W = 2, 4, 8, 8
C_out, kH, kW = 8, 3, 3

x         = torch.randn(B, C_in, H, W)
dw_weight = torch.randn(C_in, 1, kH, kW)   # one kernel per input channel
pw_weight = torch.randn(C_out, C_in, 1, 1)  # 1x1 conv

out = depthwise_separable_conv(x, dw_weight, pw_weight)
print("Output shape:", out.shape)  # (2, 8, 6, 6)

# Verify channel independence of depthwise step
patches = x.unfold(2, kH, 1).unfold(3, kW, 1)
dw_ch0 = (patches[:, 0:1] * dw_weight[0:1, 0].view(1, 1, 1, 1, kH, kW)).sum(dim=(-2, -1))
dw_ch1 = (patches[:, 1:2] * dw_weight[1:2, 0].view(1, 1, 1, 1, kH, kW)).sum(dim=(-2, -1))
print("DW ch0 and ch1 are independent (cross-correlation ~0):",
      (dw_ch0 * dw_ch1).mean().abs().item() < 1.0)

In [ ]:
from torch_judge import check
check("depthwise_conv")